# AI-Orchestrator — Geração do dataset SFT (Fase 1 LoRA) — v3

Destila 3000 exemplos do `qwen3:30b-a3b` (A100 40GB, ~6–8h). Sem upload manual: projeto baixado de suasalada.com.br.

Runtime: **A100 GPU**. Rode as células em ordem.

In [ ]:
# 1) Drive + projeto (zip servido pelo gateway via Cloudflare Tunnel)
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/AI-Orchestrator && mkdir -p /content/AI-Orchestrator
!curl -fsSL -o /content/ai-orchestrator.zip https://suasalada.com.br/ai-orchestrator.zip
!unzip -q /content/ai-orchestrator.zip -d /content/AI-Orchestrator
!ls /content/AI-Orchestrator

In [ ]:
# 2) Dependências Python
!pip install -q "httpx>=0.27" "fastapi>=0.115,<1" "uvicorn>=0.34,<1"

In [ ]:
# 3) Ollama + modelo (download ~19 GB, alguns minutos)
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd
!which zstd
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, os
os.environ['OLLAMA_KEEP_ALIVE'] = '60m'
subprocess.Popen(['ollama', 'serve'], stdout=open('/tmp/ollama.log','w'), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull qwen3:30b-a3b
# sanity: geração curta deve responder em poucos segundos na A100
!time ollama run qwen3:30b-a3b 'Responda apenas: ok'

In [ ]:
# 4) Microsserviços (4x FastAPI + SQLite, seed automático no startup)
import subprocess, time, os, json, urllib.request
SVC = {'financas': 8101, 'rh': 8102, 'estoque': 8103, 'vendas': 8104}
os.makedirs('/content/data', exist_ok=True)
for name, port in SVC.items():
    env = dict(os.environ, INTERNAL_API_KEY='colab-internal-key', DB_PATH=f'/content/data/{name}.db')
    subprocess.Popen(
        ['python', '-m', 'uvicorn', f'{name}.main:app', '--host', '127.0.0.1', '--port', str(port)],
        cwd='/content/AI-Orchestrator/services', env=env,
        stdout=open(f'/tmp/{name}.log','w'), stderr=subprocess.STDOUT)
time.sleep(6)
for name, port in SVC.items():
    body = urllib.request.urlopen(f'http://127.0.0.1:{port}/health', timeout=5).read().decode()
    print(name, body)

In [ ]:
# 5) Geração completa (~6–8h). Log em /tmp/build.log; célula segura a reconexão:
#    se a sessão cair, re-rode 1–4 e esta célula — pipeline é retomável por hash.
import subprocess, os
env = dict(os.environ,
    OLLAMA_URL='http://127.0.0.1:11434',
    MODEL='qwen3:30b-a3b',
    KEEP_ALIVE='60m',
    INTERNAL_API_KEY='colab-internal-key',
    SERVICE_URL_FINANCAS='http://127.0.0.1:8101',
    SERVICE_URL_RH='http://127.0.0.1:8102',
    SERVICE_URL_ESTOQUE='http://127.0.0.1:8103',
    SERVICE_URL_VENDAS='http://127.0.0.1:8104')
proc = subprocess.Popen(
    ['python', 'train/build_dataset.py', '--stage', 'all', '--target', '3000'],
    cwd='/content/AI-Orchestrator', env=env,
    stdout=open('/tmp/build.log','w'), stderr=subprocess.STDOUT)
print('PID', proc.pid)

In [ ]:
# 6) Monitorar (re-rode quando quiser) + backup incremental pro Drive
!tail -15 /tmp/build.log
!mkdir -p /content/drive/MyDrive/ai-orchestrator-dataset
!cp -f /content/AI-Orchestrator/train/dataset/*.jsonl /content/drive/MyDrive/ai-orchestrator-dataset/ 2>/dev/null || echo 'sem jsonl ainda'
!wc -l /content/AI-Orchestrator/train/dataset/*.jsonl 2>/dev/null || true

In [ ]:
# 7) Ao final: cópia definitiva pro Drive
!cp -f /content/AI-Orchestrator/train/dataset/*.jsonl /content/drive/MyDrive/ai-orchestrator-dataset/
!ls -la /content/drive/MyDrive/ai-orchestrator-dataset/
print('Dataset salvo no Drive — pode encerrar a sessão.')